# 18 — Multi-timeframe trendline trading simulation

Trades fitted support/resistance lines from several timeframes at once. **Long** breaks *up* through
a resistance line, **short** breaks *down* through a support line. Stop = the previous `STOP_N`-bar
extreme; exit = breakeven at 1R, target at 3R.

Edit the config below, run all, look at the chart. Everything else is in the appendix.

> Trendline breaks have already been measured here and carry **no directional edge** — the hit rate
> cleared 0.50 in 36% of 102 cells, *below* chance ([`docs/WHAT_WORKS.md`](../docs/WHAT_WORKS.md)).
> This notebook shows you what a rule did; it is not evidence that it works.


## Config

Every setting lives in [`nb18_config.py`](nb18_config.py) next to this notebook — **edit it there**.

It is a plain file, not a notebook cell, for one reason: this notebook is regenerated from a script,
and the script used to hardcode every value, so adding one setting quietly re-imposed all the
others. Nothing regenerates that file. It also means Jupyter cannot overwrite your settings when it
saves, which is the other way they have gone missing.

Edited it while the kernel was running? Re-run this cell — it reloads.

In [1]:
import sys, os, importlib, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd

# A kernel that imported `quant` before a feature existed keeps the OLD module in memory while
# Jupyter shows you the NEW file, so the traceback points at code that is not running.
import quant.scenarios.entries as _e
assert hasattr(_e, "tl_per_tf"), "Kernel -> Restart: quant was imported before this feature"

for _p in (os.getcwd(), os.path.join(os.getcwd(), "notebooks"), os.path.abspath("..")):
    if os.path.isfile(os.path.join(_p, "nb18_config.py")) and _p not in sys.path:
        sys.path.insert(0, _p)
import nb18_config as CFG
importlib.reload(CFG)                       # so an edit lands without restarting the kernel
globals().update({k: v for k, v in vars(CFG).items() if k.isupper()})

# A setting APPENDED to the config is invisible to an editor tab you already had open: saving that
# stale buffer writes the old tail back and the new line is gone. That is how EMAS_ON disappeared.
# So a setting added later also has a DEFAULT here. An older config file still runs, and the
# notebook says which line to paste in rather than dying on a NameError two cells later.
SETTING_DEFAULTS = {"EMAS_ON": True}
_missing = {k: v for k, v in SETTING_DEFAULTS.items() if not hasattr(CFG, k)}
globals().update(_missing)
if _missing:
    print("not in your config, using the default -- paste into nb18_config.py to change it:")
    for k, v in _missing.items():
        print(f"    {k} = {v!r}")

print(f"{SYMBOL} {TF}  {VIEW_START} -> {END}   lines from {TF}+{'+'.join(LINE_TFS)}")
print(f"settings from {CFG.__file__}")

BTCUSDT 5m  2026-06-01 -> 2026-06-10   lines from 5m+15m+1h
settings from C:\Users\Talal Zahid\Desktop\Talal\Binance\notebooks\nb18_config.py


## Run

In [2]:
from quant.data import get_ohlcv, tf_frames
from quant.indicators import add_mtf_emas
from quant.indicators.trendlines import mtf_trendlines
from quant.scenarios import run_pieces
from quant.scenarios.entries import trendline_trigger
from quant.strategies.conditions import raw
from quant.strategies.rulespec import build_rules, describe_rules, required_emas
from quant.viz import mtf_ema_timeframes

df = get_ohlcv(SYMBOL, TF, start=WARMUP_START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)
FRAMES = tf_frames(SYMBOL, LINE_TFS, view_start=VIEW_START, end=END,
                   warmup_bars=TF_WARMUP_BARS, source=SOURCE) if LINE_TFS else {}
# An EMA of a timeframe FINER than the chart needs its own bars: resampling only works downward,
# so a "1m" EMA derived from the 5m chart comes back byte-identical to the 5m one. These frames are
# kept OUT of FRAMES on purpose -- FRAMES is what lines are fitted on, and adding one would trade it.
_ema_only = [k for k in mtf_ema_timeframes(MTF_EMAS, chart_tf=TF) if k not in FRAMES]
EMA_FRAMES = tf_frames(SYMBOL, _ema_only, view_start=VIEW_START, end=END,
                       warmup_bars=TF_WARMUP_BARS, source=SOURCE) if _ema_only else {}

D = mtf_trendlines(df, FRAMES, **TRENDLINE)
# only the EMAs the rules actually name, on the frame they name them for
D = add_mtf_emas(D, FRAMES, required_emas(ENTRY_RULES, chart_tf=TF))

TF_NAMES = {"": TF, **{f"{k}__": k for k in FRAMES}}
PARAMS = dict(tl_prefixes=tuple(TF_NAMES), tl_tf_names=TF_NAMES,
              tl_close_n=TL_CLOSE_N, tl_full_m=TL_FULL_M, tl_touch_k=TL_TOUCH_K,
              tl_dir_n=TL_DIR_N, tl_dir_src=TL_DIR_SRC,
              tl_dir_full_n=TL_DIR_FULL_N, tl_extend=TL_EXTEND,
              tl_combine=TL_COMBINE, tl_tol_atr=TL_TOL_ATR,
              stop_n=STOP_N, stop_buffer_pct=STOP_BUFFER_PCT, be_at_rr=BE_AT_RR, tp_rr=TP_RR,
              sec_touch_n=SEC_TOUCH_N, sec_wrong_m=SEC_WRONG_M, sec_opp_n=SEC_OPP_N,
              sec_min_pct=SEC_MIN_PCT, sec_replace_tp=SEC_REPLACE_TP)
SECONDARY = "entry_retest_bail@v1" if USE_RETEST_BAIL else None

# the warm-up feeds the lines and the stop lookback; it is not traded
_vs = pd.Timestamp(VIEW_START, tz="UTC")
IN_WINDOW = raw(lambda d: (pd.to_datetime(d["t"]) >= _vs).to_numpy(), "in_window")

def _filter(side):
    # None means "no rule for this side", which is NOT the same as "nothing passes"
    extra = build_rules(ENTRY_RULES, side, chart_tf=TF, combine=RULES_COMBINE)
    return IN_WINDOW if extra is None else (IN_WINDOW & extra)

sides = [s for s, on in (("long", TRADE_LONG), ("short", TRADE_SHORT)) if on]
RUNS = {s: run_pieces(D, f"mtf_trendline_{s}@v1", "high_of_n@v1", "be_ladder@v1",
                      secondary=SECONDARY, params=PARAMS, cfg=COSTS,
                      entry_filter=_filter(s)) for s in sides}
TRADES = pd.concat([trendline_trigger(r.df, r.trades, PARAMS, prefix_names=TF_NAMES)
                    for r in RUNS.values()], ignore_index=True) \
           .sort_values("entry_time").reset_index(drop=True)

_risk = (TRADES.entry_price - TRADES.stop_price).abs()
TRADES["R"] = TRADES.pnl / (_risk * TRADES.qty)
TRADES["risk_%"] = 100 * _risk / TRADES.entry_price
TRADES["tid"] = [f"#{i+1:02d}" for i in range(len(TRADES))]
assert not len(TRADES) or TRADES.entry_time.min() >= _vs

if ENTRY_RULES:
    print(describe_rules(ENTRY_RULES, chart_tf=TF).to_string(index=False))
    print()
print(f"{len(df):,} {TF} bars | " + " ".join(f"{k} {len(v):,}" for k, v in FRAMES.items())
      + f"\n{len(TRADES)} trades   {TRADES.R.sum():+.1f}R   "
        f"{100*(TRADES.pnl>0).mean():.0f}% win   {100*TRADES.pnl.sum()/COSTS['initial_cash']:+.2f}%")

[04:26:52] INFO | quant.data | cache HIT  binance BTCUSDT 5m -> pushdown load
[04:26:53] INFO | quant.data | cache HIT  binance BTCUSDT 1h -> pushdown load
[04:26:53] INFO | quant.data | cache HIT  binance BTCUSDT 15m -> pushdown load
[04:26:53] INFO | quant.data | cache HIT  binance BTCUSDT 1m -> pushdown load
[04:26:54] INFO | quant.engine | backtest done | bars=11,521 trades=5 final_cash=9682.85 return=-3.17% dd=4.00% | 56.0 ms
[04:26:54] INFO | quant.engine | backtest done | bars=11,521 trades=6 final_cash=10893.72 return=8.94% dd=2.59% | 1.4 ms


11,521 5m bars | 1h 1,033 15m 1,729
11 trades   +4.6R   45% win   +5.77%


## Chart

| drawn | |
|---|---|
| ▲ ▼ | entry — hover for R, close reason and the line it broke |
| **grey** | the entry level |
| **red dotted** | the stop, **as set at entry** |
| **green dotted** | the 3R target |
| ✕ | the exit, green if it made money |

Five dropdowns: timeframe → pattern → occurrence, then **longs/shorts/winners/losers → one trade**.
Picking a trade clears the chart to that trade and the line that caused it.

A row tagged **`sl@be`** was stopped out on a stop that had already walked to breakeven — so its
exit sits at the entry level, nowhere near the red line, and the loss is only the fees. `ae` is the
retest bail-out.

**The EMAs stay on when you isolate a trade** — they are the context you judge it against, so the
trade picker never takes them away.

In [3]:
from quant.viz import pattern_figure

fig = pattern_figure(
    D, patterns=[], trendlines=True, trendline_kwargs=TRENDLINE, tf_frames=FRAMES,
    view_start=VIEW_START, emas=EMAS, mtf_emas=MTF_EMAS, chart_tf=TF,
    ema_frames=EMA_FRAMES, emas_on=EMAS_ON,
    sr_zones=SHOW_SR_ZONES, zone_lookback_bars=ZONE_LOOKBACK,
    max_picker_instances=MAX_PICKER, max_shapes_per_tf=MAX_SHAPES_TF,
    max_trade_picker=MAX_TRADE_PICKER,
    trades=TRADES, trade_tp_rr=TP_RR, start_visible=("trades", "sr_zones"),
    title=f"{SYMBOL} {TF}  {VIEW_START} to {END}  ·  "
          f"{'/'.join((TF,) + tuple(FRAMES))} trendline breaks  ·  {len(TRADES)} trades",
    height=HEIGHT)
# fig

Bigger, in its own browser tab, and it survives the notebook being closed:

In [4]:
from quant.viz import open_in_browser
open_in_browser(fig)          # or fig.write_html("chart.html", include_plotlyjs="cdn")

WindowsPath('C:/Users/Talal Zahid/Desktop/Talal/Binance/notebooks/chart.html')

## Results

In [5]:
by_side = TRADES.groupby("side").agg(
    trades=("R", "size"), win=("pnl", lambda z: f"{100*(z>0).mean():.0f}%"),
    total_R=("R", "sum"), avg_R=("R", "mean"))
by_why = TRADES.groupby("close_reason").agg(n=("R", "size"), total_R=("R", "sum"))
by_line = TRADES.groupby("trigger").agg(n=("R", "size"), total_R=("R", "sum"),
                                        win=("pnl", lambda z: (z > 0).mean()))
for name, t in (("by side", by_side), ("why it ended", by_why), ("by timeframe", by_line)):
    print(f"\n{name}\n{t.round(2).to_string()}")

cols = ["tid", "entry_time", "side", "cause", "entry_price", "stop_price", "exit_price",
        "risk_%", "R", "close_reason"]
TRADES[cols].round(3)


by side
       trades  win  total_R  avg_R
side                              
long        5  20%    -4.63  -0.93
short       6  67%     9.22   1.54

why it ended
                  n  total_R
close_reason                
forced_close_end  1     0.33
stop_loss         6    -7.12
take_profit       4    11.38

by timeframe
         n  total_R   win
trigger                  
15m      4     7.36  0.75
1h       1    -1.05  0.00
5m       6    -1.71  0.33


,tid,entry_time,side,cause,entry_price,stop_price,exit_price,risk_%,R,close_reason
0,#01,2026-06-01 01:40:00+00:00,short,15m·S4,73486.975,74129.046,71571.497,0.874,2.803,take_profit
1,#02,2026-06-01 19:25:00+00:00,long,5m·R55,71632.353,71346.309,71335.607,0.399,-1.437,stop_loss
2,#03,2026-06-02 07:00:00+00:00,short,15m·S5,69829.504,70759.092,67050.797,1.331,2.871,take_profit
3,#04,2026-06-03 13:10:00+00:00,long,5m·R57,67132.078,66749.259,66739.246,0.570,-1.306,stop_loss
4,#05,2026-06-04 12:30:00+00:00,long,5m·R60,63502.384,62173.898,62164.571,2.092,-1.083,stop_loss
5,#06,2026-06-05 01:00:00+00:00,short,5m·S49,63061.389,64009.989,60224.623,1.504,2.887,take_profit
6,#07,2026-06-05 18:50:00+00:00,short,1h·S7,59198.539,61182.246,61191.423,3.351,-1.053,stop_loss
7,#08,2026-06-05 22:50:00+00:00,long,15m·R8,61974.685,61177.216,61168.040,1.287,-1.135,stop_loss
8,#09,2026-06-06 06:00:00+00:00,long,5m·R66,61162.513,59470.250,61820.726,2.767,0.331,forced_close_end
9,#10,2026-06-08 12:00:00+00:00,short,5m·S53,62857.480,63905.017,63914.602,1.667,-1.106,stop_loss


---

## Appendix

### How many candles to wait — where `TL_CLOSE_N` came from

`experiments/trendline_wait_search.py` sweeps the wait per timeframe and side over **5.5 months in
6 windows**, scoring by **windows positive** rather than total R, because one huge trade in one
window is exactly what a single total hides. Nine days would give a handful of trades per cell, and
picking the argmax of six settings on ten trades is how this project has produced every result it
later had to retract.

| tf | side | trades at the pick | windows + | total R | believe it? |
|---|---|---|---|---|---|
| **5m** | long | 53 | 4/6 | −2.0R | **yes, partly** — R improves monotonically from wait 1 (−33.8R) to 4 |
| **5m** | short | 54 | 4/6 | +1.3R | same shape, −8.0R at wait 1 → +1.3R at 4 |
| 15m | short | 19 | 4/6 | +9.4R | thin |
| 15m | long | 35 | 3/6 | −9.2R | negative at every wait |
| 1h | either | 9–11 | 1–3/6 | — | **no** — too few trades to say anything |

**Nothing cleared 6 of 6.** Six settings over six windows produce a 6/6 winner by luck about 9% of
the time, so a best-of-4/6 is entirely ordinary. What is worth something is the **monotone trend on
5m**: waiting longer improves both total R *and* per-trade R (−0.32 → −0.08 for longs), so it is not
just "fewer trades paying fewer fees". That is why the 5m number is 4 and why the 1h number is set
to match rather than to its own noisy argmax.

The other thing the sweep says loudly: **longs lose at every wait on every timeframe** and shorts do
not. That is a downtrending sample, not an edge.

Re-run it yourself, or narrow it:

```
python experiments/trendline_wait_search.py
python experiments/trendline_wait_search.py --tf 5m --waits 1 2 3 4 5 6 --windows 8
```

### The cost floor

A tight stop means a large position for the same 1% risk, and fees scale with position size — so a
round trip can cost a large fraction of what you are risking. Run this to see it on your own trades:

```python
fee_R = TRADES.fees / ((TRADES.entry_price - TRADES.stop_price).abs() * TRADES.qty)
c = float(fee_R.median())
print(f"median risk {TRADES['risk_%'].median():.3f}% of price | cost {c:.2f}R per round trip")
print(f"break-even win rate p* = (1+{c:.2f})/(1+{TP_RR}) = {(1+c)/(1+TP_RR):.1%}"
      f"   vs actual {100*(TRADES.pnl>0).mean():.1f}%")
```

### The knobs

| | |
|---|---|
| `TL_CLOSE_N` | closes beyond the line before entering. Int, or a dict per timeframe with `"*"` as the fallback |
| `TL_FULL_M` | candles *entirely* beyond it, wick included. Much stricter |
| `TL_DIR_N` | candles of the trade's own colour — **red for a short, green for a long**. Not the same test as `TL_CLOSE_N`: a candle can close through a support and still be green |
| `TL_DIR_SRC` | whose candles `TL_DIR_N` counts. `"chart"` = the frame you execute on; `"line"` = the timeframe the line was fitted on, so a 15m line can demand three red *15m* candles |
| `TL_DIR_FULL_N` | candles that are **both** the right colour **and** wholly clear of the line, wicks included. Under `TL_COMBINE="all"` this equals setting `TL_FULL_M` and `TL_DIR_N` to the same number; as one condition it also survives `"any"` |
| `TL_EXTEND` | a line is retired the moment it breaks, so a multi-bar rule needs to know where it *would* have been. `True` carries it along its own slope; `False` holds it flat. A rising support keeps rising, so flat is wrong on one side and lenient on the other |
| `TL_TOUCH_K` | only trade lines that had been respected this many times |
| `TL_COMBINE` | `"all"` intersects the enabled conditions, `"any"` unions them |
| `STOP_N` | stop lookback. **Larger = wider stop = smaller position = less cost in R** |
| `BE_AT_RR` / `TP_RR` | the exit ladder. `BE_AT_RR=0` disables the breakeven move |
| `USE_RETEST_BAIL` | if price keeps returning to the entry, give up on the target and exit on the first opposite candle while green |
| `SEC_TOUCH_N` / `SEC_WRONG_M` | what arms it: bars that traded back through the entry, or candles formed wholly on its losing side. Counted **per trade** in the kernel, since the level is that trade's own entry |
| `SEC_MIN_PCT` | how far price must have moved your way before the bail-out may fire. **Set it above your round trip** — the engine's own gate is "any favourable move", which exits below cost about a third of the time here |
| `EMAS` / `MTF_EMAS` | moving averages, and they stay visible while one trade is isolated |
| `ENTRY_RULES` | extra conditions, one sentence each, ANDed onto the break. They can only remove entries |
| `RULES_COMBINE` | `"all"` needs every rule for that side, `"any"` needs one |

### The rule language

```
<n>  <reference>  <close | formed fully | touched>  [above | below]  [red | green | NA]
```

| | |
|---|---|
| `EMA_5m_200` | the 200 EMA of 5m. On a 5m chart that is `ema_200`; on a 1m chart `5m__ema_200` |
| `RES_15m` / `SUP_15m` | the fitted trendline of that timeframe |
| `close` | the candle's **close** is above/below the level |
| `formed fully` | the **whole** candle is, wicks included — much stricter, and not interchangeable: a candle can close below a support with its high still above it |
| `touched` | the candle's range **contains** the level. Neither above nor below, so it takes no side |
| `long:` / `short:` | restricts a rule to that side; without a prefix it applies to both |

Word order does not matter and filler words are ignored, so `"red 3 the EMA_5m_200 is formed fully
below"` parses the same as `"3 candles EMA_15m_50 formed fully below red"`. A sentence that cannot
be read raises an error naming the problem rather than quietly matching nothing.

Colour is literal — `red` means red on both sides. The direction-relative version (red for a short,
green for a long) is `TL_DIR_N`, which is part of the entry itself.

One thing to watch: with `max_open_trades=1`, filtering one side can **increase** the total trade
count, because a slot the filtered trade would have occupied is now free for the other side.
| `LINE_TFS` | which timeframes contribute lines. `()` trades `TF` lines only |

### What would make this fair rather than plausible

None of it is done here: difference against a within-day-shuffled null
(`experiments/null_control.py`); beat the `time_entry` metronome through the same exits; count
survivors against `n_configs × 0.5^n_windows`; and trim the top K trades, not just the top one.
